# 12 — Data Cleaning & Normalization

---

## Objective

Clean, normalize, and quality-check **all raw data** from the extraction pipeline,
then route each output to the appropriate processed folder.

### Processing Pipeline

```
data/raw/                                  data/processed/
├── bigdata/bigquery/  ──────┐             ├── legitimate/
├── bigdata/common_crawl/ ───┤             │   └── legitimate_clean_<N>.csv
├── csv/  ───────────────────┤             │
├── scraping/certfr/  ───────┤── clean ──→ ├── adapted/
├── db/adapted_fr_phishing ──┤             │   └── adapted_clean_<N>.csv
├── db/synthetic_fr_emails ──┘             │
                                           └── synthetic/
                                               └── synthetic_clean_<N>.csv
```

### Output Structure

| Folder | Content | Sources |
|--------|---------|--------|
| `data/processed/legitimate/` | Cleaned legitimate (ham) emails | CSV sources, synthetic legit, BigQuery legit (if any) |
| `data/processed/adapted/` | Cleaned adapted FR phishing | Notebook 10 (EN→FR cultural adaptation) |
| `data/processed/synthetic/` | Cleaned synthetic FR phishing | Notebook 11 (synthetic generation) |

### Unified Schema

All output CSVs use the same columns:

| Column | Type | Description |
|--------|------|-------------|
| `text` | str | Cleaned email body (PII anonymized) |
| `label` | int | 0 = legitimate, 1 = phishing |
| `source` | str | Origin dataset identifier |
| `language` | str | ISO 639-1 language code (fr/en) |
| `archetype` | str | French phishing archetype (if applicable) |
| `text_len` | int | Character count of cleaned text |

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
import html
import os
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException

# Ensure CWD is project root (2 levels up from notebooks/processing/)
os.chdir(Path(__file__).resolve().parent.parent.parent if "__file__" in dir() else Path.cwd())
while not (Path.cwd() / "pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir(Path.cwd().parent)
print(f"Working dir: {Path.cwd()}")

# ── Paths ──────────────────────────────────────────────────────────────
RAW_DIR: Path = Path("data/raw")
PROC_DIR: Path = Path("data/processed")

# Output folders matching the required structure
LEGIT_DIR: Path = PROC_DIR / "legitimate"
ADAPTED_DIR: Path = PROC_DIR / "adapted"
SYNTHETIC_DIR: Path = PROC_DIR / "synthetic"

for d in [LEGIT_DIR, ADAPTED_DIR, SYNTHETIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Thresholds ──────────────────────────────────────────────────────────
MIN_TEXT_LEN: int = 30       # Minimum chars to keep
MAX_TEXT_LEN: int = 10_000   # Maximum chars (truncate beyond)
DEDUP_HASH_LEN: int = 300   # Hash first N chars for dedup

TIMESTAMP: str = datetime.now(timezone.utc).strftime("%Y%m%d")

print(f"Raw dir      : {RAW_DIR.resolve()}")
print(f"Processed dir: {PROC_DIR.resolve()}")
print(f"  legitimate : {LEGIT_DIR}")
print(f"  adapted    : {ADAPTED_DIR}")
print(f"  synthetic  : {SYNTHETIC_DIR}")
print(f"Min text     : {MIN_TEXT_LEN} chars")
print(f"Max text     : {MAX_TEXT_LEN} chars")

Working dir: /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre
Raw dir      : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw
Processed dir: /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/processed
  legitimate : data/processed/legitimate
  adapted    : data/processed/adapted
  synthetic  : data/processed/synthetic
Min text     : 30 chars
Max text     : 10000 chars


## 1. Cleaning Functions

Shared cleaning pipeline applied to all text data regardless of source.

In [2]:
# ── PII Anonymization (RGPD) ─────────────────────────────────────────

# Email addresses
_RE_EMAIL = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
# French phone numbers (01-09 prefix, various separators)
_RE_PHONE_FR = re.compile(r'\b0[1-9][ .-]?(?:\d{2}[ .-]?){4}\b')
# International phone numbers with +
_RE_PHONE_INTL = re.compile(r'\+\d{1,3}[\s.-]?\d{1,4}[\s.-]?(?:\d{2,4}[\s.-]?){2,4}')
# French IBAN
_RE_IBAN = re.compile(r'\bFR\d{2}[\s]?(?:\d{4}[\s]?){5}\d{3}\b')
# Sécu number (1 or 2 + 13 digits)
_RE_SECU = re.compile(r'\b[12]\s?\d{2}\s?\d{2}\s?\d{2}\s?\d{3}\s?\d{3}\s?\d{2}\b')
# SIRET (14 digits, possibly with spaces)
_RE_SIRET = re.compile(r'\b\d{3}\s?\d{3}\s?\d{3}\s?\d{5}\b')
# URLs
_RE_URL = re.compile(r'https?://[^\s<>"\')]+|www\.[^\s<>"\')]+', re.IGNORECASE)


def anonymize_pii(text: str) -> str:
    """Replace PII patterns with anonymization tokens (RGPD compliance)."""
    text = _RE_EMAIL.sub('[EMAIL]', text)
    text = _RE_IBAN.sub('[IBAN]', text)
    text = _RE_SECU.sub('[SECU]', text)
    text = _RE_SIRET.sub('[SIRET]', text)
    text = _RE_PHONE_INTL.sub('[PHONE]', text)
    text = _RE_PHONE_FR.sub('[PHONE]', text)
    text = _RE_URL.sub('[URL]', text)
    return text


# ── HTML & Formatting Cleanup ─────────────────────────────────────────
_RE_HTML_TAGS = re.compile(r'<[^>]+>')
_RE_MULTI_NEWLINE = re.compile(r'\n{3,}')
_RE_MULTI_SPACE = re.compile(r'[ \t]{2,}')
_RE_NON_PRINTABLE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')


def clean_text(text: str) -> str:
    """Clean a single text field: strip HTML, normalize whitespace, anonymize PII."""
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # 1. Decode HTML entities
    text = html.unescape(text)
    
    # 2. Strip HTML tags
    text = _RE_HTML_TAGS.sub(' ', text)
    
    # 3. Remove non-printable characters (keep newlines/tabs)
    text = _RE_NON_PRINTABLE.sub('', text)
    
    # 4. Normalize Unicode (NFC — canonical decomposition + composition)
    text = unicodedata.normalize('NFC', text)
    
    # 5. Normalize whitespace
    text = _RE_MULTI_SPACE.sub(' ', text)
    text = _RE_MULTI_NEWLINE.sub('\n\n', text)
    text = text.strip()
    
    # 6. Anonymize PII (RGPD)
    text = anonymize_pii(text)
    
    # 7. Truncate if too long
    if len(text) > MAX_TEXT_LEN:
        text = text[:MAX_TEXT_LEN] + '…'
    
    return text


def detect_language(text: str) -> str:
    """Detect language with fallback."""
    try:
        return detect(text[:500])  # Use first 500 chars for speed
    except LangDetectException:
        return "unknown"


# ── Test cleaning ─────────────────────────────────────────────────────
test_dirty: str = (
    '<p>Bonjour M. Dupont,</p>\n\n\n\n'
    'Contactez-nous à contact@example.com ou 01 23 45 67 89.\n'
    'IBAN: FR76 3000 6000 0112 3456 7890 189\n'
    'Lien: https://phishing-site.example.com/login'
)
test_clean: str = clean_text(test_dirty)
print("BEFORE:")
print(test_dirty)
print("\nAFTER:")
print(test_clean)
print(f"\nCleaning functions loaded. PII patterns: 7")

BEFORE:
<p>Bonjour M. Dupont,</p>



Contactez-nous à contact@example.com ou 01 23 45 67 89.
IBAN: FR76 3000 6000 0112 3456 7890 189
Lien: https://phishing-site.example.com/login

AFTER:
Bonjour M. Dupont, 

Contactez-nous à [EMAIL] ou [PHONE].
IBAN: [IBAN]
Lien: [URL]

Cleaning functions loaded. PII patterns: 7


## 2. Load All Raw Data

In [3]:
# ── Load raw CSVs ─────────────────────────────────────────────────────

def load_latest_csv(directory: Path, pattern: str = "*.csv") -> pd.DataFrame | None:
    """Load the most recent CSV matching pattern in a directory."""
    csvs = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        return None
    path = csvs[0]
    df = pd.read_csv(path)
    print(f"  Loaded: {path.name} ({len(df):,} rows)")
    return df


raw_sources: dict[str, pd.DataFrame] = {}

# ── 1. BigQuery EN phishing ──────────────────────────────────────────
print("\n[bigdata/bigquery]")
df_bq = load_latest_csv(RAW_DIR / "bigdata" / "bigquery", "bigquery_phishing_*.csv")
if df_bq is not None:
    raw_sources["bigquery_en_phishing"] = df_bq

# ── 2. Common Crawl FR pages ─────────────────────────────────────────
print("\n[bigdata/common_crawl]")
df_cc = load_latest_csv(RAW_DIR / "bigdata" / "common_crawl", "common_crawl_fr_usable_*.csv")
if df_cc is not None:
    raw_sources["common_crawl_fr"] = df_cc

# ── 3. CSV sources (HF datasets) ────────────────────────────────────
print("\n[csv]")
df_csv = load_latest_csv(RAW_DIR / "csv", "csv_sources_combined_*.csv")
if df_csv is not None:
    raw_sources["csv_hf_legit"] = df_csv

# ── 4. CERT-FR scraping (all reports) ────────────────────────────────
print("\n[scraping/certfr]")
df_certfr = load_latest_csv(RAW_DIR / "scraping" / "certfr", "certfr_cti_reports_*.csv")
if df_certfr is not None:
    raw_sources["certfr_reports"] = df_certfr

# ── 5. Adapted FR phishing (notebook 10) ─────────────────────────────
print("\n[db/adapted]")
adapted_path = RAW_DIR / "db" / "adapted_fr_phishing.csv"
if adapted_path.exists():
    df_adapted = pd.read_csv(adapted_path)
    raw_sources["adapted_en_fr"] = df_adapted
    print(f"  Loaded: adapted_fr_phishing.csv ({len(df_adapted):,} rows)")
else:
    print("  ⚠ Not found — run notebook 10 first")

# ── 6. Synthetic FR emails (notebook 11) ─────────────────────────────
print("\n[db/synthetic]")
synthetic_path = RAW_DIR / "db" / "synthetic_fr_emails.csv"
if synthetic_path.exists():
    df_synthetic = pd.read_csv(synthetic_path)
    raw_sources["synthetic_fr"] = df_synthetic
    print(f"  Loaded: synthetic_fr_emails.csv ({len(df_synthetic):,} rows)")
else:
    print("  ⚠ Not found — run notebook 11 first")

# ── 7. PhishTank (API) — optional ────────────────────────────────────
print("\n[api/phishtank]")
df_pt = load_latest_csv(RAW_DIR / "api" / "phishtank", "phishtank_*.csv")
if df_pt is not None:
    raw_sources["phishtank_urls"] = df_pt
else:
    print("  ⚠ Not available (API rate-limited)")

# ── Summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Total raw sources loaded: {len(raw_sources)}")
for name, df in raw_sources.items():
    print(f"  {name:30s} : {len(df):>7,} rows")


[bigdata/bigquery]
  Loaded: bigquery_phishing_en_4597_20260228.csv (4,597 rows)

[bigdata/common_crawl]
  Loaded: common_crawl_fr_usable_28_20260228.csv (28 rows)

[csv]
  Loaded: csv_sources_combined_6606_20260301.csv (6,606 rows)

[scraping/certfr]
  Loaded: certfr_cti_reports_91_20260301.csv (91 rows)

[db/adapted]
  Loaded: adapted_fr_phishing.csv (2,400 rows)

[db/synthetic]
  Loaded: synthetic_fr_emails.csv (2,863 rows)

[api/phishtank]
  ⚠ Not available (API rate-limited)

Total raw sources loaded: 6
  bigquery_en_phishing           :   4,597 rows
  common_crawl_fr                :      28 rows
  csv_hf_legit                   :   6,606 rows
  certfr_reports                 :      91 rows
  adapted_en_fr                  :   2,400 rows
  synthetic_fr                   :   2,863 rows


## 3. Process Each Source

Apply cleaning + normalization, then route to the correct output folder.

In [4]:
# ── 3a. Process ADAPTED data (→ data/processed/adapted/) ─────────────
# Source: adapted_fr_phishing.csv from notebook 10

print("Processing ADAPTED data...")

if "adapted_en_fr" in raw_sources:
    df = raw_sources["adapted_en_fr"].copy()
    before: int = len(df)
    
    # Identify text column
    text_col: str = "text" if "text" in df.columns else df.columns[0]
    
    # Clean text
    df["text"] = df[text_col].astype(str).apply(clean_text)
    
    # Normalize label to int
    if "label" in df.columns:
        df["label"] = df["label"].astype(int)
    else:
        df["label"] = 1  # All adapted emails are phishing
    
    # Ensure required columns
    df["source"] = df.get("source", "adapted_en_fr")
    df["language"] = "fr"
    df["archetype"] = df.get("archetype", "")
    df["text_len"] = df["text"].str.len()
    
    # Filter: min length + non-empty
    df = df[df["text_len"] >= MIN_TEXT_LEN].reset_index(drop=True)
    
    # Deduplicate
    df["_hash"] = df["text"].str[:DEDUP_HASH_LEN].apply(
        lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
    )
    df = df.drop_duplicates(subset="_hash", keep="first").drop(columns="_hash").reset_index(drop=True)
    
    # Select output columns
    output_cols: list[str] = ["text", "label", "source", "language", "archetype", "text_len"]
    df_adapted_clean = df[[c for c in output_cols if c in df.columns]]
    
    # Export
    out_path = ADAPTED_DIR / f"adapted_clean_{len(df_adapted_clean)}_{TIMESTAMP}.csv"
    df_adapted_clean.to_csv(out_path, index=False, encoding="utf-8")
    
    print(f"  Before     : {before:,}")
    print(f"  After clean: {len(df_adapted_clean):,}")
    print(f"  Exported   : {out_path}")
    print(f"  Size       : {out_path.stat().st_size / 1024:.1f} KB")
    print(f"  Labels     : {dict(df_adapted_clean['label'].value_counts())}")
    print(f"  Archetypes : {dict(df_adapted_clean['archetype'].value_counts())}")
else:
    df_adapted_clean = pd.DataFrame()
    print("  ⚠ No adapted data available")

Processing ADAPTED data...
  Before     : 2,400
  After clean: 2,145
  Exported   : data/processed/adapted/adapted_clean_2145_20260301.csv
  Size       : 992.7 KB
  Labels     : {1: np.int64(2145)}
  Archetypes : {'dgfip_tax': np.int64(300), 'caf_allocation': np.int64(300), 'laposte_colis': np.int64(300), 'facture_paiement': np.int64(300), 'ameli_sante': np.int64(299), 'franceconnect_id': np.int64(270), 'banque_securite': np.int64(223), 'urssaf_cotisation': np.int64(153)}


In [5]:
# ── 3b. Process SYNTHETIC data (→ data/processed/synthetic/) ──────────
# Source: synthetic_fr_emails.csv from notebook 11
# Contains BOTH phishing (label=1) and legitimate (label=0)
# We split: phishing → synthetic/, legit → collected for legitimate/ later

print("Processing SYNTHETIC data...")

synthetic_legit_rows: list[pd.DataFrame] = []

if "synthetic_fr" in raw_sources:
    df = raw_sources["synthetic_fr"].copy()
    before = len(df)
    
    text_col = "text" if "text" in df.columns else df.columns[0]
    
    # Clean text
    df["text"] = df[text_col].astype(str).apply(clean_text)
    
    # Normalize label
    if "label" in df.columns:
        df["label"] = df["label"].astype(int)
    
    df["source"] = df.get("source", "synthetic_fr")
    df["language"] = "fr"
    df["archetype"] = df.get("archetype", "")
    df["text_len"] = df["text"].str.len()
    
    # Filter min length
    df = df[df["text_len"] >= MIN_TEXT_LEN].reset_index(drop=True)
    
    # Deduplicate
    df["_hash"] = df["text"].str[:DEDUP_HASH_LEN].apply(
        lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
    )
    df = df.drop_duplicates(subset="_hash", keep="first").drop(columns="_hash").reset_index(drop=True)
    
    output_cols = ["text", "label", "source", "language", "archetype", "text_len"]
    df_clean = df[[c for c in output_cols if c in df.columns]]
    
    # Split: phishing → synthetic/, legit → collect for legitimate/
    df_synth_phishing = df_clean[df_clean["label"] == 1].reset_index(drop=True)
    df_synth_legit = df_clean[df_clean["label"] == 0].reset_index(drop=True)
    
    # Export phishing to synthetic/
    out_path = SYNTHETIC_DIR / f"synthetic_clean_{len(df_synth_phishing)}_{TIMESTAMP}.csv"
    df_synth_phishing.to_csv(out_path, index=False, encoding="utf-8")
    
    print(f"  Before     : {before:,}")
    print(f"  After clean: {len(df_clean):,} (phishing: {len(df_synth_phishing)}, legit: {len(df_synth_legit)})")
    print(f"  Phishing → : {out_path}")
    print(f"  Size       : {out_path.stat().st_size / 1024:.1f} KB")
    print(f"  Legit collected for legitimate/ folder: {len(df_synth_legit):,} rows")
    
    # Collect legit for later
    if not df_synth_legit.empty:
        synthetic_legit_rows.append(df_synth_legit)
else:
    df_synth_phishing = pd.DataFrame()
    print("  ⚠ No synthetic data available")

Processing SYNTHETIC data...
  Before     : 2,863
  After clean: 2,610 (phishing: 1747, legit: 863)
  Phishing → : data/processed/synthetic/synthetic_clean_1747_20260301.csv
  Size       : 746.1 KB
  Legit collected for legitimate/ folder: 863 rows


In [ ]:
# ── 3c. Process LEGITIMATE data (→ data/processed/legitimate/) ────────
# Sources:
#   - CSV HF sources (ham emails, label=0)
#   - Synthetic legit (label=0 from notebook 11)
#   - Common Crawl FR pages (web content as supplementary text)
#   - BigQuery legit subset (if any label=0 rows)

print("Processing LEGITIMATE data...")

legit_parts: list[pd.DataFrame] = list(synthetic_legit_rows)  # Start with synthetic legit

# Standard output columns — shared across all legitimate sub-sources
output_cols: list[str] = ["text", "label", "source", "language", "archetype", "text_len"]

# ── CSV HuggingFace sources ──────────────────────────────────────────
if "csv_hf_legit" in raw_sources:
    df = raw_sources["csv_hf_legit"].copy()
    text_col = "text" if "text" in df.columns else df.columns[0]
    
    df["text"] = df[text_col].astype(str).apply(clean_text)
    
    # Normalize labels: ham→0, spam→1
    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: 0 if str(x).lower() in ("ham", "0", "legitimate") else 1
        ).astype(int)
    else:
        df["label"] = 0
    
    # Keep only legitimate (label=0)
    df = df[df["label"] == 0].reset_index(drop=True)
    
    df["source"] = df.get("source", "csv_hf")
    df["language"] = df.get("language", "en")
    df["archetype"] = ""
    df["text_len"] = df["text"].str.len()
    df = df[df["text_len"] >= MIN_TEXT_LEN].reset_index(drop=True)
    
    legit_parts.append(df[[c for c in output_cols if c in df.columns]])
    print(f"  CSV HF     : {len(df):,} legit rows")

# ── BigQuery (legitimate subset if any) ──────────────────────────────
if "bigquery_en_phishing" in raw_sources:
    df = raw_sources["bigquery_en_phishing"].copy()
    text_col = "text" if "text" in df.columns else "content" if "content" in df.columns else df.columns[0]
    
    # Check for legitimate emails
    if "label" in df.columns:
        df_bq_legit = df[df["label"] == 0].copy()
        if not df_bq_legit.empty:
            df_bq_legit["text"] = df_bq_legit[text_col].astype(str).apply(clean_text)
            df_bq_legit["label"] = 0
            df_bq_legit["source"] = "bigquery_en"
            df_bq_legit["language"] = "en"
            df_bq_legit["archetype"] = ""
            df_bq_legit["text_len"] = df_bq_legit["text"].str.len()
            df_bq_legit = df_bq_legit[df_bq_legit["text_len"] >= MIN_TEXT_LEN].reset_index(drop=True)
            legit_parts.append(df_bq_legit[[c for c in output_cols if c in df_bq_legit.columns]])
            print(f"  BigQuery   : {len(df_bq_legit):,} legit rows")
        else:
            print("  BigQuery   : 0 legit (all phishing, as expected)")
    else:
        print("  BigQuery   : no label column, skipping")

# ── Common Crawl FR pages (supplementary text) ────────────────────────
if "common_crawl_fr" in raw_sources:
    df = raw_sources["common_crawl_fr"].copy()
    # CC data has different columns — find text-like column
    text_col = next(
        (c for c in ["url", "title", "text", "content"] if c in df.columns),
        df.columns[0]
    )
    print(f"  Common Crawl: {len(df)} pages (metadata, not email bodies — skipping for email pipeline)")

# ── CERT-FR reports (threat intel, not email bodies) ──────────────────
if "certfr_reports" in raw_sources:
    df = raw_sources["certfr_reports"]
    print(f"  CERT-FR    : {len(df)} reports (CTI metadata — not email bodies, skipping for email pipeline)")

# ── Combine all legitimate parts ─────────────────────────────────────
if legit_parts:
    df_legit_combined = pd.concat(legit_parts, ignore_index=True)
    
    # Dedup
    before = len(df_legit_combined)
    df_legit_combined["_hash"] = df_legit_combined["text"].str[:DEDUP_HASH_LEN].apply(
        lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
    )
    df_legit_combined = df_legit_combined.drop_duplicates(
        subset="_hash", keep="first"
    ).drop(columns="_hash").reset_index(drop=True)
    
    out_path = LEGIT_DIR / f"legitimate_clean_{len(df_legit_combined)}_{TIMESTAMP}.csv"
    df_legit_combined.to_csv(out_path, index=False, encoding="utf-8")
    
    print(f"\n  Combined legit: {before:,} → {len(df_legit_combined):,} (after dedup)")
    print(f"  Exported   : {out_path}")
    print(f"  Size       : {out_path.stat().st_size / 1024:.1f} KB")
    print(f"  Sources    : {dict(df_legit_combined['source'].value_counts())}")
    print(f"  Languages  : {dict(df_legit_combined['language'].value_counts())}")
else:
    df_legit_combined = pd.DataFrame()
    print("  ⚠ No legitimate data available")

Processing LEGITIMATE data...
  CSV HF     : 6,601 legit rows
  BigQuery   : 0 legit (all phishing, as expected)
  Common Crawl: 28 pages (metadata, not email bodies — skipping for email pipeline)
  CERT-FR    : 91 reports (CTI metadata — not email bodies, skipping for email pipeline)

  Combined legit: 7,464 → 7,461 (after dedup)
  Exported   : data/processed/legitimate/legitimate_clean_7461_20260301.csv
  Size       : 11469.3 KB
  Sources    : {'cybersectony_phishing_v2': np.int64(6598), 'synthetic_fr': np.int64(863)}
  Languages  : {'en': np.int64(6598), 'fr': np.int64(863)}


## 4. Quality Report

In [7]:
# ── Quality Report ────────────────────────────────────────────────────
print("=" * 70)
print("PROCESSING SUMMARY")
print("=" * 70)

summary: list[dict] = []

for folder, label_name, df in [
    (ADAPTED_DIR, "adapted", df_adapted_clean if 'df_adapted_clean' in dir() else pd.DataFrame()),
    (SYNTHETIC_DIR, "synthetic", df_synth_phishing if 'df_synth_phishing' in dir() else pd.DataFrame()),
    (LEGIT_DIR, "legitimate", df_legit_combined if 'df_legit_combined' in dir() else pd.DataFrame()),
]:
    if df.empty:
        continue
    
    phishing_count = int((df["label"] == 1).sum()) if "label" in df.columns else 0
    legit_count = int((df["label"] == 0).sum()) if "label" in df.columns else 0
    
    row = {
        "folder": label_name,
        "rows": len(df),
        "phishing": phishing_count,
        "legit": legit_count,
        "avg_len": int(df["text_len"].mean()) if "text_len" in df.columns else 0,
        "min_len": int(df["text_len"].min()) if "text_len" in df.columns else 0,
        "max_len": int(df["text_len"].max()) if "text_len" in df.columns else 0,
    }
    summary.append(row)
    
    print(f"\n{label_name.upper()}/")
    print(f"  Rows       : {row['rows']:,}")
    print(f"  Phishing   : {row['phishing']:,}")
    print(f"  Legitimate : {row['legit']:,}")
    print(f"  Text length: avg={row['avg_len']}, min={row['min_len']}, max={row['max_len']}")

df_summary = pd.DataFrame(summary)
total_rows = df_summary["rows"].sum()
total_phishing = df_summary["phishing"].sum()
total_legit = df_summary["legit"].sum()

print(f"\n{'='*70}")
print(f"TOTAL: {total_rows:,} rows (phishing: {total_phishing:,}, legit: {total_legit:,})")
print(f"Phishing ratio: {total_phishing / total_rows * 100:.1f}%" if total_rows > 0 else "")
print(f"{'='*70}")

display(df_summary)

PROCESSING SUMMARY

ADAPTED/
  Rows       : 2,145
  Phishing   : 2,145
  Legitimate : 0
  Text length: avg=417, min=302, max=657

SYNTHETIC/
  Rows       : 1,747
  Phishing   : 1,747
  Legitimate : 0
  Text length: avg=382, min=306, max=519

LEGITIMATE/
  Rows       : 7,461
  Phishing   : 0
  Legitimate : 7,461
  Text length: avg=1530, min=32, max=10001

TOTAL: 11,353 rows (phishing: 3,892, legit: 7,461)
Phishing ratio: 34.3%


,folder,rows,phishing,legit,avg_len,min_len,max_len
0,adapted,2145,2145,0,417,302,657
1,synthetic,1747,1747,0,382,306,519
2,legitimate,7461,0,7461,1530,32,10001


In [8]:
# ── PII Anonymization Verification ────────────────────────────────────
# Spot-check that PII patterns were removed

print("PII Anonymization Check")
print("=" * 50)

for name, df in [
    ("adapted", df_adapted_clean if 'df_adapted_clean' in dir() else pd.DataFrame()),
    ("synthetic", df_synth_phishing if 'df_synth_phishing' in dir() else pd.DataFrame()),
    ("legitimate", df_legit_combined if 'df_legit_combined' in dir() else pd.DataFrame()),
]:
    if df.empty:
        continue
    
    texts = df["text"].dropna()
    all_text = texts.str.cat(sep=" ")
    
    # Check for residual PII
    email_leaks = len(_RE_EMAIL.findall(all_text))
    phone_leaks = len(_RE_PHONE_FR.findall(all_text))
    iban_leaks = len(_RE_IBAN.findall(all_text))
    
    # Count anonymization tokens
    email_tokens = all_text.count("[EMAIL]")
    phone_tokens = all_text.count("[PHONE]")
    url_tokens = all_text.count("[URL]")
    
    status = "✓ CLEAN" if (email_leaks + phone_leaks + iban_leaks) == 0 else "⚠ LEAKS"
    print(f"\n{name}/ — {status}")
    print(f"  [EMAIL] tokens: {email_tokens}  (raw leaks: {email_leaks})")
    print(f"  [PHONE] tokens: {phone_tokens}  (raw leaks: {phone_leaks})")
    print(f"  [URL]   tokens: {url_tokens}")
    print(f"  [IBAN]  leaks : {iban_leaks}")

PII Anonymization Check

adapted/ — ✓ CLEAN
  [EMAIL] tokens: 0  (raw leaks: 0)
  [PHONE] tokens: 163  (raw leaks: 0)
  [URL]   tokens: 2050
  [IBAN]  leaks : 0

synthetic/ — ✓ CLEAN
  [EMAIL] tokens: 0  (raw leaks: 0)
  [PHONE] tokens: 31  (raw leaks: 0)
  [URL]   tokens: 1586
  [IBAN]  leaks : 0

legitimate/ — ✓ CLEAN
  [EMAIL] tokens: 2363  (raw leaks: 0)
  [PHONE] tokens: 238  (raw leaks: 0)
  [URL]   tokens: 4890
  [IBAN]  leaks : 0


In [9]:
# ── Verify output file structure ──────────────────────────────────────
print("Output File Structure")
print("=" * 50)

for folder in [LEGIT_DIR, ADAPTED_DIR, SYNTHETIC_DIR]:
    files = sorted(folder.glob("*.csv"))
    print(f"\n{folder}/")
    if files:
        for f in files:
            size = f.stat().st_size / 1024
            print(f"  {f.name} ({size:.1f} KB)")
    else:
        print("  (empty)")

Output File Structure

data/processed/legitimate/
  legitimate_clean_7461_20260301.csv (11469.3 KB)

data/processed/adapted/
  adapted_clean_2145_20260301.csv (992.7 KB)

data/processed/synthetic/
  synthetic_clean_1747_20260301.csv (746.1 KB)


## 5. Summary

### Processing Pipeline

| Step | Action |
|------|--------|
| 1. Load | Read all CSVs from `data/raw/` source folders |
| 2. Clean text | Strip HTML, normalize Unicode (NFC), collapse whitespace |
| 3. Anonymize PII | Replace emails, phones, IBANs, sécu numbers, SIRETs, URLs |
| 4. Normalize labels | Map to int: 0 = legitimate, 1 = phishing |
| 5. Filter | Remove texts < 30 chars, truncate > 10K chars |
| 6. Deduplicate | SHA-256 hash of first 300 chars |
| 7. Route | Adapted → `processed/adapted/`, Synthetic phishing → `processed/synthetic/`, All legit → `processed/legitimate/` |

### Output Structure

```
data/processed/
├── adapted/              ← Culturally-adapted FR phishing (from NB10)
│   └── adapted_clean_<N>_<date>.csv
├── synthetic/            ← Synthetic FR phishing (from NB11)
│   └── synthetic_clean_<N>_<date>.csv
└── legitimate/           ← All legitimate emails (CSV sources + synthetic legit)
    └── legitimate_clean_<N>_<date>.csv
```

### RGPD Compliance

| PII Type | Regex Pattern | Token |
|----------|---------------|-------|
| Email addresses | `[A-Za-z0-9._%+-]+@...` | `[EMAIL]` |
| French phone | `0[1-9] XX XX XX XX` | `[PHONE]` |
| International phone | `+XX XXXX...` | `[PHONE]` |
| French IBAN | `FRXX XXXX...` | `[IBAN]` |
| Sécu number | `[12] XX XX XX XXX XXX XX` | `[SECU]` |
| SIRET | `XXX XXX XXX XXXXX` | `[SIRET]` |
| URLs | `https?://...` | `[URL]` |

### Next Step

The cleaned data in `data/processed/` feeds the aggregation notebook which:
1. Merges all three folders into a balanced training set
2. Creates train/val/test splits (70/15/15)
3. Saves to `data/final/` for CamemBERTv2 fine-tuning